In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import sys
import numpy as np
import xarray as xr


SWOOSH_FILE = "/mnt/soclim0/public_data/weiji/swoosh/swoosh-v02.72-198401-202601-lonlatpress-20deg-5deg-L31.nc"
TEMPLATE_FILE = "/mnt/soclim0/andreas/vmro3_input4MIPs_ozone_CMIP6_UReading-CCMI_2020.nc"


def open_dataset_safely(path):
    engines = [None, "netcdf4", "h5netcdf", "scipy"]
    last_err = None
    for eng in engines:
        try:
            kwargs = dict(decode_times=False, mask_and_scale=True)
            if eng is None:
                ds = xr.open_dataset(path, **kwargs)
                return ds, "default"
            else:
                ds = xr.open_dataset(path, engine=eng, **kwargs)
                return ds, eng
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to open {path}\nLast error: {last_err}")


def format_attrs(attrs, keys=None):
    if not attrs:
        return "{}"
    if keys is None:
        keys = list(attrs.keys())
    out = []
    for k in keys:
        if k in attrs:
            out.append(f"{k}={attrs[k]}")
    return "; ".join(out) if out else "{}"


def print_header(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def summarize_coords(ds, name):
    print_header(f"{name}: COORDINATES / DIMS")
    print(f"Dimensions: {dict(ds.sizes)}")
    print(f"Coordinates: {list(ds.coords)}")

    for cname in ds.coords:
        da = ds[cname]
        vals = da.values
        print(f"\n[{cname}]")
        print(f"  dims   : {da.dims}")
        print(f"  shape  : {da.shape}")
        print(f"  dtype  : {da.dtype}")
        print(f"  attrs  : {format_attrs(da.attrs, ['long_name', 'standard_name', 'units', 'calendar', 'axis', 'positive'])}")

        if vals.ndim == 1 and vals.size > 0 and np.issubdtype(vals.dtype, np.number):
            head = vals[:min(5, vals.size)]
            tail = vals[-min(5, vals.size):]
            print(f"  first  : {head}")
            print(f"  last   : {tail}")
            try:
                diffs = np.diff(vals.astype(float))
                if diffs.size > 0:
                    print(f"  diff   : min={np.nanmin(diffs):.6g}, max={np.nanmax(diffs):.6g}")
            except Exception:
                pass


def summarize_data_vars(ds, name, max_show=30):
    print_header(f"{name}: DATA VARIABLES")
    vars_all = list(ds.data_vars)
    print(f"Total data variables: {len(vars_all)}")

    for v in vars_all[:max_show]:
        da = ds[v]
        print(f"\n[{v}]")
        print(f"  dims   : {da.dims}")
        print(f"  shape  : {da.shape}")
        print(f"  dtype  : {da.dtype}")
        print(f"  attrs  : {format_attrs(da.attrs, ['long_name', 'standard_name', 'units', 'grid_type', 'cell_methods', 'missing_value', '_FillValue'])}")

    if len(vars_all) > max_show:
        print(f"\n... ({len(vars_all) - max_show} more variables not shown)")


def find_ozone_candidates(ds):
    out = []
    for v in ds.data_vars:
        da = ds[v]
        text = " ".join([
            v.lower(),
            str(da.attrs.get("long_name", "")).lower(),
            str(da.attrs.get("standard_name", "")).lower(),
            str(da.attrs.get("units", "")).lower(),
        ])
        if ("o3" in text) or ("ozone" in text):
            out.append(v)
    return out


def print_ozone_candidates(ds, name):
    print_header(f"{name}: OZONE CANDIDATES")
    cands = find_ozone_candidates(ds)
    if not cands:
        print("No ozone-like variables found.")
        return []

    for v in cands:
        da = ds[v]
        print(f"\n[{v}]")
        print(f"  dims   : {da.dims}")
        print(f"  shape  : {da.shape}")
        print(f"  dtype  : {da.dtype}")
        print(f"  attrs  : {format_attrs(da.attrs, ['long_name', 'standard_name', 'units', 'grid_type', 'missing_value', '_FillValue'])}")
    return cands


def inspect_var_sample(ds, varname, label):
    if varname not in ds:
        print(f"{label}: variable {varname} not found")
        return

    da = ds[varname]
    print_header(f"{label}: SAMPLE CHECK FOR {varname}")
    print(f"dims   : {da.dims}")
    print(f"shape  : {da.shape}")
    print(f"dtype  : {da.dtype}")
    print(f"attrs  : {format_attrs(da.attrs, ['long_name', 'standard_name', 'units', 'grid_type', 'missing_value', '_FillValue'])}")

    # Avoid loading full array
    try:
        sample = da
        indexers = {}
        for d in da.dims:
            n = da.sizes[d]
            indexers[d] = min(n // 2, n - 1)
        one = da.isel(**indexers).values
        print(f"single-point sample at mid index: {one}")
    except Exception as e:
        print(f"single-point sample failed: {e}")

    # first-time slice stats (still modest)
    try:
        if "time" in da.dims:
            sample2 = da.isel(time=0)
        else:
            sample2 = da

        # reduce very large 3D/4D sample further for speed if needed
        vals = sample2.values
        finite = np.isfinite(vals)
        nfinite = int(finite.sum())
        ntotal = vals.size
        print(f"slice finite count: {nfinite}/{ntotal}")
        if nfinite > 0:
            print(f"slice min/max     : {np.nanmin(vals):.6g}, {np.nanmax(vals):.6g}")
    except Exception as e:
        print(f"slice stats failed: {e}")


def compare_coord(ds1, ds2, c1, c2):
    if (c1 not in ds1.coords) or (c2 not in ds2.coords):
        print(f"  coord compare skipped: {c1} or {c2} missing")
        return

    a = ds1[c1].values
    b = ds2[c2].values

    print(f"\nCompare coord: {c1} (SWOOSH) vs {c2} (TEMPLATE)")
    print(f"  shapes : {a.shape} vs {b.shape}")
    print(f"  dtypes : {a.dtype} vs {b.dtype}")

    try:
        same_len = len(a) == len(b)
        print(f"  same length: {same_len}")
    except Exception:
        pass

    if np.ndim(a) == 1 and np.ndim(b) == 1 and a.size > 0 and b.size > 0:
        try:
            print(f"  range A: {np.nanmin(a):.6g} .. {np.nanmax(a):.6g}")
            print(f"  range B: {np.nanmin(b):.6g} .. {np.nanmax(b):.6g}")
        except Exception:
            pass

        try:
            if len(a) == len(b):
                diff = np.nanmax(np.abs(a.astype(float) - b.astype(float)))
                print(f"  max abs diff (same length only): {diff:.6g}")
        except Exception:
            pass


def infer_main_var(ds, preferred_patterns):
    vars_all = list(ds.data_vars)
    for pat in preferred_patterns:
        for v in vars_all:
            if re.fullmatch(pat, v):
                return v
    for pat in preferred_patterns:
        for v in vars_all:
            if re.search(pat, v):
                return v
    return None


def main():
    print_header("OPEN FILES")
    print(f"SWOOSH   : {SWOOSH_FILE}")
    print(f"TEMPLATE : {TEMPLATE_FILE}")

    if not os.path.exists(SWOOSH_FILE):
        raise FileNotFoundError(SWOOSH_FILE)
    if not os.path.exists(TEMPLATE_FILE):
        raise FileNotFoundError(TEMPLATE_FILE)

    ds_sw, eng_sw = open_dataset_safely(SWOOSH_FILE)
    ds_tp, eng_tp = open_dataset_safely(TEMPLATE_FILE)

    print(f"Opened SWOOSH with engine   : {eng_sw}")
    print(f"Opened TEMPLATE with engine : {eng_tp}")

    print_header("GLOBAL ATTRIBUTES")
    print("[SWOOSH]")
    print(format_attrs(ds_sw.attrs))
    print("\n[TEMPLATE]")
    print(format_attrs(ds_tp.attrs))

    summarize_coords(ds_sw, "SWOOSH")
    summarize_coords(ds_tp, "TEMPLATE")

    summarize_data_vars(ds_sw, "SWOOSH", max_show=20)
    summarize_data_vars(ds_tp, "TEMPLATE", max_show=20)

    sw_o3 = print_ozone_candidates(ds_sw, "SWOOSH")
    tp_o3 = print_ozone_candidates(ds_tp, "TEMPLATE")

    # Very likely choices
    sw_main = infer_main_var(ds_sw, [r"combinedo3q", r".*o3.*"])
    tp_main = infer_main_var(ds_tp, [r"vmro3", r".*o3.*", r".*ozone.*"])

    print_header("LIKELY MAIN VARIABLE GUESS")
    print(f"SWOOSH   main guess : {sw_main}")
    print(f"TEMPLATE main guess : {tp_main}")

    if sw_main is not None:
        inspect_var_sample(ds_sw, sw_main, "SWOOSH")
    if tp_main is not None:
        inspect_var_sample(ds_tp, tp_main, "TEMPLATE")

    print_header("COORDINATE NAME GUESS")
    print(f"SWOOSH coords   : {list(ds_sw.coords)}")
    print(f"TEMPLATE coords : {list(ds_tp.coords)}")

    # Common likely comparisons
    for cands in [("lon", "lon"), ("lat", "lat"), ("level", "plev"), ("level", "lev"), ("time", "time")]:
        compare_coord(ds_sw, ds_tp, cands[0], cands[1])

    print_header("AUTOMATIC INTERPRETATION")
    if sw_main is None:
        print("Could not identify SWOOSH main ozone variable automatically.")
    else:
        print(f"1) SWOOSH ozone source is very likely: {sw_main}")

    if tp_main is None:
        print("2) Could not identify template ozone variable automatically.")
    else:
        print(f"2) Template target variable is very likely: {tp_main}")

    if sw_main is not None and tp_main is not None:
        sw_units = str(ds_sw[sw_main].attrs.get("units", ""))
        tp_units = str(ds_tp[tp_main].attrs.get("units", ""))
        print(f"3) Units: SWOOSH={sw_units!r}, TEMPLATE={tp_units!r}")

        if sw_units.lower() == "ppmv" and ("mol" in tp_units.lower() or tp_units.strip() in ["1", "1.0"]):
            print("   -> Very likely need unit conversion: ppmv -> mol/mol via x 1e-6")
        elif sw_units == tp_units:
            print("   -> Units already match")
        else:
            print("   -> Check unit conversion carefully")

        print(f"4) SWOOSH dims   : {ds_sw[sw_main].dims}")
        print(f"5) TEMPLATE dims : {ds_tp[tp_main].dims}")
        print("   -> If dim names/order differ, rename/reorder to match template")
        print("   -> If coordinate values differ, interpolation/regridding is needed")
        print("   -> If template is yearly (e.g. 2020 only), final output likely needs one file per year")

    print_header("NEXT STEP")
    print("Run this script first. Then paste the full output back.")
    print("Once we see the template variable name, units, dims, and coordinate values,")
    print("we can write the actual conversion script with high confidence.")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import math
import numpy as np
import xarray as xr

SWOOSH_FILE = "/mnt/soclim0/public_data/weiji/swoosh/swoosh-v02.72-198401-202601-lonlatpress-20deg-5deg-L31.nc"
TEMPLATE_FILE = "/mnt/soclim0/andreas/vmro3_input4MIPs_ozone_CMIP6_UReading-CCMI_2020.nc"

SW_VAR = "combinedo3q"
TP_VAR = "vmro3"


def print_header(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def ym_from_month_index(month_index, base_year=1850, base_month=1):
    """
    Convert integer month index since base_year-base_month to (year, month).
    Example: 0 -> 1850-01, 1 -> 1850-02, ...
    """
    total = (base_year * 12 + (base_month - 1)) + int(month_index)
    year = total // 12
    month = total % 12 + 1
    return year, month


def infer_template_year_months(time_vals):
    """
    Template time units are 'months since 1850-01-01 00:00:00'.
    The numeric values are month-midpoint-like values, so floor(time)
    should correspond to the month index.
    """
    out = []
    for i, t in enumerate(time_vals):
        month_index = math.floor(float(t))
        y, m = ym_from_month_index(month_index, base_year=1850, base_month=1)
        out.append((i, float(t), month_index, y, m))
    return out


def infer_bounds_year_months(bounds_vals):
    """
    bounds_vals shape: (time, 2)
    Infer start/end year-month by flooring each bound.
    For monthly files this is usually enough to identify the month window.
    """
    out = []
    for i in range(bounds_vals.shape[0]):
        b0 = float(bounds_vals[i, 0])
        b1 = float(bounds_vals[i, 1])
        idx0 = math.floor(b0)
        idx1 = math.floor(b1)
        y0, m0 = ym_from_month_index(idx0, base_year=1850, base_month=1)
        y1, m1 = ym_from_month_index(idx1, base_year=1850, base_month=1)
        out.append((i, b0, b1, idx0, y0, m0, idx1, y1, m1))
    return out


def summarize_bounds_var(ds, varname):
    if varname not in ds:
        print(f"{varname}: not found")
        return
    da = ds[varname]
    print(f"{varname}: dims={da.dims}, shape={da.shape}, dtype={da.dtype}")
    try:
        vals = da.values
        print(f"  first rows:\n{vals[:5]}")
        print(f"  last rows:\n{vals[-5:]}")
    except Exception as e:
        print(f"  unable to print values: {e}")


def safe_open(path):
    engines = [None, "netcdf4", "h5netcdf", "scipy"]
    last_err = None
    for eng in engines:
        try:
            kwargs = dict(decode_times=False, mask_and_scale=True)
            if eng is None:
                ds = xr.open_dataset(path, **kwargs)
                return ds, "default"
            ds = xr.open_dataset(path, engine=eng, **kwargs)
            return ds, eng
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Cannot open {path}. Last error: {last_err}")


def finite_stats_for_time_slice(da):
    vals = da.values
    finite = np.isfinite(vals)
    n_total = vals.size
    n_finite = int(finite.sum())
    frac = n_finite / n_total if n_total > 0 else np.nan

    if n_finite > 0:
        vmin = float(np.nanmin(vals))
        vmax = float(np.nanmax(vals))
    else:
        vmin = np.nan
        vmax = np.nan

    return n_finite, n_total, frac, vmin, vmax


def main():
    print_header("OPEN FILES")
    print(f"SWOOSH   : {SWOOSH_FILE}")
    print(f"TEMPLATE : {TEMPLATE_FILE}")

    if not os.path.exists(SWOOSH_FILE):
        raise FileNotFoundError(SWOOSH_FILE)
    if not os.path.exists(TEMPLATE_FILE):
        raise FileNotFoundError(TEMPLATE_FILE)

    ds_sw, eng_sw = safe_open(SWOOSH_FILE)
    ds_tp, eng_tp = safe_open(TEMPLATE_FILE)

    print(f"Opened SWOOSH with engine   : {eng_sw}")
    print(f"Opened TEMPLATE with engine : {eng_tp}")

    print_header("BASIC VARIABLE CHECK")
    print(f"SWOOSH variable present   ({SW_VAR}) : {SW_VAR in ds_sw}")
    print(f"TEMPLATE variable present ({TP_VAR}) : {TP_VAR in ds_tp}")

    print_header("MONTHLY-MEAN CHECK")
    print(f"TEMPLATE global frequency : {ds_tp.attrs.get('frequency', 'N/A')}")
    if TP_VAR in ds_tp:
        print(f"TEMPLATE {TP_VAR} cell_methods : {ds_tp[TP_VAR].attrs.get('cell_methods', 'N/A')}")
        print(f"TEMPLATE {TP_VAR} units        : {ds_tp[TP_VAR].attrs.get('units', 'N/A')}")
    print(f"SWOOSH time units              : {ds_sw['time'].attrs.get('units', 'N/A')}")
    print(f"SWOOSH year/month vars present : {'year' in ds_sw and 'month' in ds_sw}")
    print("Interpretation: template is monthly mean; SWOOSH is monthly gridded data indexed by year/month.")

    print_header("GRID-BOUNDARY VARIABLES")
    print("SWOOSH boundary-like vars:")
    summarize_bounds_var(ds_sw, "latbound")
    summarize_bounds_var(ds_sw, "lonbound")

    print("\nTEMPLATE boundary vars:")
    summarize_bounds_var(ds_tp, "lat_bnds")
    summarize_bounds_var(ds_tp, "lon_bnds")
    summarize_bounds_var(ds_tp, "plev_bnds")
    summarize_bounds_var(ds_tp, "time_bnds")

    print("\nInterpretation:")
    print("- SWOOSH latbound/lonbound are source-grid boundary metadata.")
    print("- TEMPLATE lat_bnds/lon_bnds/plev_bnds/time_bnds are target-grid boundary metadata.")
    print("- For interpolation we mainly use center coords (lat/lon/plev), not SWOOSH latbound.")
    print("- Final output should follow the TEMPLATE bounds convention, not copy SWOOSH latbound directly.")

    print_header("TEMPLATE TIME AXIS")
    tvals = ds_tp["time"].values
    print(f"template time raw values ({len(tvals)}):")
    print(tvals)

    inferred = infer_template_year_months(tvals)
    print("\nInferred template year-month from floor(time):")
    print("idx | raw_time | month_index | year-month")
    for i, raw_t, month_index, y, m in inferred:
        print(f"{i:2d} | {raw_t:10.8f} | {month_index:4d} | {y:04d}-{m:02d}")

    if "time_bnds" in ds_tp:
        tb = ds_tp["time_bnds"].values
        print("\nTemplate time_bnds raw values:")
        print(tb)

        inferred_b = infer_bounds_year_months(tb)
        print("\nInferred template time bounds year-month:")
        print("idx | b0 | b1 | start_idx | start_ym | end_idx | end_ym")
        for row in inferred_b:
            i, b0, b1, idx0, y0, m0, idx1, y1, m1 = row
            print(
                f"{i:2d} | {b0:10.8f} | {b1:10.8f} | "
                f"{idx0:4d} | {y0:04d}-{m0:02d} | "
                f"{idx1:4d} | {y1:04d}-{m1:02d}"
            )

    print_header("SWOOSH AVAILABLE YEAR-MONTHS")
    sw_year = ds_sw["year"].values.astype(int)
    sw_month = ds_sw["month"].values.astype(int)
    sw_time = ds_sw["time"].values

    print(f"SWOOSH year range : {sw_year.min()}-{sw_month[sw_year.argmin()]:02d} to {sw_year.max()}-{sw_month[sw_year.argmax()]:02d}")
    print("First 12 SWOOSH months:")
    for i in range(min(12, len(sw_year))):
        print(f"{i:3d}: {sw_year[i]:04d}-{sw_month[i]:02d}  raw_time={sw_time[i]:8.3f}")

    print("\nLast 12 SWOOSH months:")
    for i in range(max(0, len(sw_year) - 12), len(sw_year)):
        print(f"{i:3d}: {sw_year[i]:04d}-{sw_month[i]:02d}  raw_time={sw_time[i]:8.3f}")

    print_header("MATCH TEMPLATE MONTHS TO SWOOSH MONTHS")
    sw_map = {}
    for i, (y, m) in enumerate(zip(sw_year, sw_month)):
        sw_map[(int(y), int(m))] = i

    missing = []
    matched_rows = []

    for idx, raw_t, month_index, y, m in inferred:
        sw_idx = sw_map.get((y, m), None)
        if sw_idx is None:
            missing.append((idx, y, m))
            matched_rows.append((idx, y, m, None, None, None, None, None))
            continue

        if SW_VAR not in ds_sw:
            matched_rows.append((idx, y, m, sw_idx, float(sw_time[sw_idx]), None, None, None))
            continue

        da_slice = ds_sw[SW_VAR].isel(time=sw_idx)
        n_finite, n_total, frac, vmin, vmax = finite_stats_for_time_slice(da_slice)
        matched_rows.append((idx, y, m, sw_idx, float(sw_time[sw_idx]), frac, vmin, vmax))

    print("tpl_idx | tpl_ym  | sw_idx | sw_time_raw | finite_frac | min | max")
    for row in matched_rows:
        idx, y, m, sw_idx, sw_raw, frac, vmin, vmax = row
        tpl_ym = f"{y:04d}-{m:02d}"
        if sw_idx is None:
            print(f"{idx:7d} | {tpl_ym} |   MISSING")
        else:
            frac_s = "N/A" if frac is None else f"{frac:.6f}"
            vmin_s = "N/A" if vmin is None or not np.isfinite(vmin) else f"{vmin:.6g}"
            vmax_s = "N/A" if vmax is None or not np.isfinite(vmax) else f"{vmax:.6g}"
            print(f"{idx:7d} | {tpl_ym} | {sw_idx:6d} | {sw_raw:11.3f} | {frac_s:11s} | {vmin_s:>8s} | {vmax_s:>8s}")

    print_header("SUMMARY")
    print(f"Template months total : {len(inferred)}")
    print(f"Missing in SWOOSH     : {len(missing)}")
    if missing:
        print("Missing template months:")
        for idx, y, m in missing:
            print(f"  template idx {idx}: {y:04d}-{m:02d}")
    else:
        print("All template months exist in SWOOSH.")

    print("\nKey interpretation:")
    print("1) `combinedo3q` should be the SWOOSH source ozone field.")
    print("2) Template `vmro3` is monthly mean ozone mole fraction.")
    print("3) This script checks whether each target template month exists in SWOOSH.")
    print("4) `finite_frac` tells you whether SWOOSH has valid data in that month.")
    print("5) `latbound` is source-grid metadata, not the final target bounds variable.")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Prepare SWOOSH combined O3 onto the CMIP6 input4MIPs template grid/format.

Main choices implemented:
- Source field: SWOOSH `combinedo3q`
- Target field: template `vmro3`
- Time handling: exact month matching to template months
- Units: ppmv -> mole mole-1 (x 1e-6)
- Horizontal interpolation: linear on lat/lon after converting lon to [0, 360)
- Vertical interpolation: linear in log-pressure space
- Outside SWOOSH vertical coverage: edge extrapolation (nearest-edge fill)
- Output coords/bounds: follow template
- Output time/time_bnds: copied from template exactly

The script prints extensive debug information at each major step.
"""

import os
import math
import json
import warnings
from typing import List, Tuple, Dict

import numpy as np
import xarray as xr

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# USER SETTINGS
# =============================================================================
SWOOSH_FILE = "/mnt/soclim0/public_data/weiji/swoosh/swoosh-v02.72-198401-202601-lonlatpress-20deg-5deg-L31.nc"
TEMPLATE_FILE = "/mnt/soclim0/andreas/vmro3_input4MIPs_ozone_CMIP6_UReading-CCMI_2020.nc"

SOURCE_VAR = "combinedo3q"
TARGET_VAR = "vmro3"

OUT_DIR = "/mnt/soclim0/public_data/weiji/swoosh/processed_like_input4MIPs"
OUT_FILE = os.path.join(OUT_DIR, "vmro3_input4MIPs_ozone_SWOOSH_like_template_201912-202101.nc")
DEBUG_JSON = os.path.join(OUT_DIR, "vmro3_input4MIPs_ozone_SWOOSH_like_template_201912-202101.debug.json")

# Set to None to use all template times.
# Or e.g. TARGET_TEMPLATE_YEAR = 2020 to keep months from Dec(prev year) to Jan(next year)
TARGET_TEMPLATE_YEAR = None

# chunk sizes can help if dask is available; harmless if not
OPEN_KW = dict(decode_times=False, mask_and_scale=True)

# =============================================================================
# HELPERS
# =============================================================================
def print_header(title: str) -> None:
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)

def safe_open_dataset(path: str) -> Tuple[xr.Dataset, str]:
    engines = [None, "netcdf4", "h5netcdf", "scipy"]
    last_err = None
    for eng in engines:
        try:
            if eng is None:
                ds = xr.open_dataset(path, **OPEN_KW)
                return ds, "default"
            ds = xr.open_dataset(path, engine=eng, **OPEN_KW)
            return ds, eng
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Cannot open {path}\nLast error: {last_err}")

def infer_template_year_months(time_vals: np.ndarray) -> List[Tuple[int, float, int, int, int]]:
    """
    Template time units are months since 1850-01-01.
    floor(time) identifies the target month index.
    Returns: (idx, raw_time, month_index, year, month)
    """
    out = []
    for i, t in enumerate(time_vals):
        month_index = int(math.floor(float(t)))
        total_months = 1850 * 12 + month_index
        year = total_months // 12
        month = total_months % 12 + 1
        out.append((i, float(t), month_index, year, month))
    return out

def finite_stats_np(arr: np.ndarray) -> Dict[str, float]:
    finite = np.isfinite(arr)
    n_total = int(arr.size)
    n_finite = int(finite.sum())
    frac = float(n_finite / n_total) if n_total else np.nan
    if n_finite:
        return {
            "n_total": n_total,
            "n_finite": n_finite,
            "finite_frac": frac,
            "min": float(np.nanmin(arr)),
            "max": float(np.nanmax(arr)),
            "mean": float(np.nanmean(arr)),
        }
    return {
        "n_total": n_total,
        "n_finite": n_finite,
        "finite_frac": frac,
        "min": np.nan,
        "max": np.nan,
        "mean": np.nan,
    }

def da_stats(da: xr.DataArray, label: str) -> Dict[str, float]:
    vals = da.values
    s = finite_stats_np(vals)
    print(f"{label}: shape={da.shape}, n_finite={s['n_finite']}/{s['n_total']}, "
          f"frac={s['finite_frac']:.6f}, min={s['min']:.6g}, max={s['max']:.6g}, mean={s['mean']:.6g}")
    return s

def convert_lon_to_360(ds: xr.Dataset, lon_name: str = "lon") -> xr.Dataset:
    lon = ds[lon_name]
    lon360 = xr.where(lon < 0, lon + 360, lon)
    ds2 = ds.assign_coords({lon_name: lon360})
    ds2 = ds2.sortby(lon_name)
    return ds2

def build_swoosh_time_lookup(ds_sw: xr.Dataset) -> Dict[Tuple[int, int], int]:
    years = ds_sw["year"].values.astype(int)
    months = ds_sw["month"].values.astype(int)
    lookup = {}
    for i, (y, m) in enumerate(zip(years, months)):
        lookup[(int(y), int(m))] = i
    return lookup

def select_template_indices_by_yearmonth(meta: List[Tuple[int, float, int, int, int]],
                                         target_year: int = None) -> List[int]:
    """
    If target_year is None, keep all.
    If target_year = 2020, keep 2019-12 through 2021-01 for this template style.
    """
    if target_year is None:
        return [i for i, *_ in meta]

    keep = []
    for i, _, _, y, m in meta:
        if (y == target_year - 1 and m == 12) or (y == target_year) or (y == target_year + 1 and m == 1):
            keep.append(i)
    return keep

def match_template_months_to_swoosh(ds_sw: xr.Dataset, ds_tp: xr.Dataset, target_year: int = None):
    tmeta_all = infer_template_year_months(ds_tp["time"].values)
    keep_template_idx = select_template_indices_by_yearmonth(tmeta_all, target_year=target_year)
    tmeta = [tmeta_all[i] for i in keep_template_idx]

    lookup = build_swoosh_time_lookup(ds_sw)
    matched = []
    missing = []
    for tpl_idx, raw_t, month_idx, y, m in tmeta:
        sw_idx = lookup.get((y, m))
        if sw_idx is None:
            missing.append((tpl_idx, y, m))
        matched.append((tpl_idx, raw_t, month_idx, y, m, sw_idx))
    return tmeta_all, keep_template_idx, matched, missing

def interp_logp_with_edge_fill(values_src: np.ndarray,
                               p_src_hpa: np.ndarray,
                               p_tgt_hpa: np.ndarray) -> np.ndarray:
    """
    values_src: (..., lev_src)
    p_src_hpa:  (lev_src,)
    p_tgt_hpa:  (lev_tgt,)
    Interpolates along last axis in log-pressure.
    Outside range uses nearest-edge fill.
    NaNs in source profile are handled by valid-point interpolation;
    if fewer than 2 valid points, returns all-NaN or edge-filled from one point.
    """
    logp_src = np.log(np.asarray(p_src_hpa, dtype=np.float64))
    logp_tgt = np.log(np.asarray(p_tgt_hpa, dtype=np.float64))

    shp = values_src.shape[:-1]
    n_prof = int(np.prod(shp)) if shp else 1
    n_src = values_src.shape[-1]
    n_tgt = len(p_tgt_hpa)

    arr2d = values_src.reshape(n_prof, n_src)
    out = np.full((n_prof, n_tgt), np.nan, dtype=np.float64)

    for i in range(n_prof):
        prof = arr2d[i, :]
        good = np.isfinite(prof) & np.isfinite(logp_src)
        ng = int(good.sum())

        if ng == 0:
            continue
        elif ng == 1:
            out[i, :] = prof[good][0]
            continue

        x = logp_src[good]
        y = prof[good]

        # ensure ascending x for np.interp
        order = np.argsort(x)
        x = x[order]
        y = y[order]

        # linear interpolation with edge fill
        yi = np.interp(logp_tgt, x, y, left=y[0], right=y[-1])
        out[i, :] = yi

    return out.reshape(*shp, n_tgt)

def make_template_like_attrs(var_template: xr.DataArray) -> Dict[str, str]:
    attrs = dict(var_template.attrs)
    attrs["long_name"] = var_template.attrs.get("long_name", TARGET_VAR)
    attrs["standard_name"] = var_template.attrs.get("standard_name", "mole_fraction_of_ozone_in_air")
    attrs["units"] = var_template.attrs.get("units", "mole mole-1")
    if "cell_methods" in var_template.attrs:
        attrs["cell_methods"] = var_template.attrs["cell_methods"]
    return attrs

# =============================================================================
# MAIN
# =============================================================================
def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    debug = {}

    print_header("1) OPEN DATASETS")
    print(f"SWOOSH   : {SWOOSH_FILE}")
    print(f"TEMPLATE : {TEMPLATE_FILE}")
    ds_sw, eng_sw = safe_open_dataset(SWOOSH_FILE)
    ds_tp, eng_tp = safe_open_dataset(TEMPLATE_FILE)
    print(f"Opened SWOOSH with engine   : {eng_sw}")
    print(f"Opened TEMPLATE with engine : {eng_tp}")

    debug["engines"] = {"swoosh": eng_sw, "template": eng_tp}

    # -------------------------
    # Basic sanity checks
    # -------------------------
    print_header("2) BASIC CHECKS")
    assert SOURCE_VAR in ds_sw, f"{SOURCE_VAR} not found in SWOOSH"
    assert TARGET_VAR in ds_tp, f"{TARGET_VAR} not found in TEMPLATE"
    for c in ["time", "lat", "lon", "level", "year", "month"]:
        assert c in ds_sw.variables or c in ds_sw.coords, f"SWOOSH missing {c}"
    for c in ["time", "lat", "lon", "plev"]:
        assert c in ds_tp.variables or c in ds_tp.coords, f"TEMPLATE missing {c}"
    print("Variable and coordinate existence checks passed.")

    print("\nSWOOSH dims:", dict(ds_sw.sizes))
    print("TEMPLATE dims:", dict(ds_tp.sizes))
    print("\nSWOOSH source variable attrs:", dict(ds_sw[SOURCE_VAR].attrs))
    print("TEMPLATE target variable attrs:", dict(ds_tp[TARGET_VAR].attrs))

    # -------------------------
    # Time matching
    # -------------------------
    print_header("3) TIME MATCHING")
    tmeta_all, keep_template_idx, matched, missing = match_template_months_to_swoosh(
        ds_sw, ds_tp, target_year=TARGET_TEMPLATE_YEAR
    )
    print("Template time mapping (idx, raw_time, month_index, year-month, sw_idx):")
    for tpl_idx, raw_t, month_idx, y, m, sw_idx in matched:
        print(f"tpl_idx={tpl_idx:2d}, raw={raw_t:10.8f}, month_index={month_idx:4d}, "
              f"ym={y:04d}-{m:02d}, sw_idx={sw_idx}")
    if missing:
        raise ValueError(f"Missing template months in SWOOSH: {missing}")

    debug["matched_months"] = [
        {
            "template_index": int(tpl_idx),
            "template_raw_time": float(raw_t),
            "template_month_index": int(month_idx),
            "year": int(y),
            "month": int(m),
            "swoosh_index": int(sw_idx),
        }
        for tpl_idx, raw_t, month_idx, y, m, sw_idx in matched
    ]

    # subset template on desired time indices
    ds_tp_sub = ds_tp.isel(time=keep_template_idx)
    print(f"\nSelected template time count: {ds_tp_sub.sizes['time']}")
    print("Selected template raw time:", ds_tp_sub["time"].values)

    # select SWOOSH in exactly matching year/month order
    sw_indices = [sw_idx for *_rest, sw_idx in matched]
    ds_sw_sub = ds_sw.isel(time=sw_indices)

    print("\nCheck SWOOSH selected year-month sequence:")
    for i in range(ds_sw_sub.sizes["time"]):
        y = int(ds_sw_sub["year"].values[i])
        m = int(ds_sw_sub["month"].values[i])
        t = float(ds_sw_sub["time"].values[i])
        print(f"selected[{i:2d}] => {y:04d}-{m:02d}, raw_time={t:.3f}")

    # Check source slice validity month by month
    print("\nSWOOSH month-by-month source-data validity:")
    source_month_stats = []
    for i in range(ds_sw_sub.sizes["time"]):
        y = int(ds_sw_sub["year"].values[i])
        m = int(ds_sw_sub["month"].values[i])
        s = da_stats(ds_sw_sub[SOURCE_VAR].isel(time=i), f"{y:04d}-{m:02d} {SOURCE_VAR}")
        s["year"] = y
        s["month"] = m
        source_month_stats.append(s)
    debug["source_month_stats"] = source_month_stats

    # -------------------------
    # Build source ozone data
    # -------------------------
    print_header("4) BUILD SOURCE OZONE FIELD")
    da_src = ds_sw_sub[SOURCE_VAR].astype(np.float64)
    print("Original source dims:", da_src.dims)
    print("Original source coords:")
    print("  time :", da_src.sizes["time"])
    print("  level:", da_src.sizes["level"], f"range {float(da_src['level'].min()):.6g}..{float(da_src['level'].max()):.6g} hPa")
    print("  lat  :", da_src.sizes["lat"],   f"range {float(da_src['lat'].min()):.6g}..{float(da_src['lat'].max()):.6g}")
    print("  lon  :", da_src.sizes["lon"],   f"range {float(da_src['lon'].min()):.6g}..{float(da_src['lon'].max()):.6g}")

    # Convert units
    da_src = da_src * 1.0e-6
    da_src.attrs["units"] = "mole mole-1"
    da_stats(da_src.isel(time=0), "After unit conversion, first month source field")

    # Convert lon to 0..360 and sort
    ds_src_work = da_src.to_dataset(name=TARGET_VAR)
    ds_src_work = convert_lon_to_360(ds_src_work, lon_name="lon")
    da_src = ds_src_work[TARGET_VAR]

    print("\nAfter lon conversion/sort:")
    print("  lon first 10:", da_src["lon"].values[:10])
    print("  lon last  10:", da_src["lon"].values[-10:])
    assert float(da_src["lon"].min()) >= 0.0 - 1e-9
    assert float(da_src["lon"].max()) <= 360.0 + 1e-9

    # -------------------------
    # Horizontal interpolation
    # -------------------------
    print_header("5) HORIZONTAL INTERPOLATION")
    tgt_lat = ds_tp_sub["lat"]
    tgt_lon = ds_tp_sub["lon"]

    print("Target lat size/range:", tgt_lat.size, float(tgt_lat.min()), float(tgt_lat.max()))
    print("Target lon size/range:", tgt_lon.size, float(tgt_lon.min()), float(tgt_lon.max()))

    # xarray interp works over coordinate centers
    da_h = da_src.interp(lat=tgt_lat, lon=tgt_lon, method="linear")
    da_stats(da_h.isel(time=0), "After horizontal interpolation, first month")
    print("After horizontal interpolation dims:", da_h.dims, da_h.shape)

    # -------------------------
    # Vertical interpolation (log-p)
    # -------------------------
    print_header("6) VERTICAL INTERPOLATION (LOG-P WITH EDGE EXTRAPOLATION)")
    p_src = da_h["level"].values.astype(np.float64)
    p_tgt = ds_tp_sub["plev"].values.astype(np.float64)

    print("Source plev count/range:", len(p_src), float(np.nanmin(p_src)), float(np.nanmax(p_src)), "hPa")
    print("Target plev count/range:", len(p_tgt), float(np.nanmin(p_tgt)), float(np.nanmax(p_tgt)), "hPa")

    print("\nHow many target levels are outside SWOOSH range?")
    psrc_min = float(np.nanmin(p_src))
    psrc_max = float(np.nanmax(p_src))
    below_src = int(np.sum(p_tgt > psrc_max))  # larger pressure = lower atmosphere
    above_src = int(np.sum(p_tgt < psrc_min))  # smaller pressure = upper atmosphere
    within_src = int(np.sum((p_tgt >= psrc_min) & (p_tgt <= psrc_max)))
    print(f"  below SWOOSH lower boundary (> {psrc_max:.6g} hPa): {below_src}")
    print(f"  above SWOOSH upper boundary (< {psrc_min:.6g} hPa): {above_src}")
    print(f"  within SWOOSH pressure range                     : {within_src}")

    # transpose to time, lat, lon, level for vectorized interpolation along last axis
    da_h_t = da_h.transpose("time", "lat", "lon", "level")
    arr_h = da_h_t.values
    arr_v = interp_logp_with_edge_fill(arr_h, p_src_hpa=p_src, p_tgt_hpa=p_tgt)

    da_v = xr.DataArray(
        arr_v,
        dims=("time", "lat", "lon", "plev"),
        coords={
            "time": ds_tp_sub["time"],
            "lat": ds_tp_sub["lat"],
            "lon": ds_tp_sub["lon"],
            "plev": ds_tp_sub["plev"],
        },
        name=TARGET_VAR,
    ).transpose("time", "plev", "lat", "lon")

    da_v.attrs = make_template_like_attrs(ds_tp_sub[TARGET_VAR])
    da_stats(da_v.isel(time=0), "After vertical interpolation, first month")
    print("After vertical interpolation dims:", da_v.dims, da_v.shape)

    # -------------------------
    # Time/bounds/metadata assembly
    # -------------------------
    print_header("7) ASSEMBLE TEMPLATE-LIKE OUTPUT DATASET")
    out = xr.Dataset()

    # Copy template coords exactly for selected time subset
    out = out.assign_coords(
        time=ds_tp_sub["time"],
        plev=ds_tp_sub["plev"],
        lat=ds_tp_sub["lat"],
        lon=ds_tp_sub["lon"],
    )

    # Carry over coordinate attrs
    for c in ["time", "plev", "lat", "lon"]:
        out[c].attrs = dict(ds_tp_sub[c].attrs)

    # Copy boundary variables exactly from template subset
    for v in ["time_bnds", "plev_bnds", "lat_bnds", "lon_bnds"]:
        if v in ds_tp_sub:
            out[v] = ds_tp_sub[v]

    # Add data
    out[TARGET_VAR] = da_v.astype(np.float32)
    out[TARGET_VAR].attrs = make_template_like_attrs(ds_tp_sub[TARGET_VAR])

    # Global attrs: start from template, then annotate provenance
    gattrs = dict(ds_tp_sub.attrs)
    old_comment = gattrs.get("comment", "")
    new_comment = (
        "Prepared from SWOOSH v02.72 combinedo3q and remapped onto the template grid/format. "
        "Units converted from ppmv to mole mole-1. Horizontal interpolation: linear. "
        "Vertical interpolation: linear in log-pressure space. "
        "Outside SWOOSH vertical coverage, nearest-edge extrapolation was applied. "
    )
    gattrs["comment"] = (old_comment + " " + new_comment).strip()
    gattrs["source"] = "SWOOSH v02.72 combinedo3q remapped to template grid"
    gattrs["history"] = "Created by prepare_swoosh_to_template.py"
    out.attrs = gattrs

    print("Output dataset summary:")
    print(out)
    print("\nOutput variable attrs:")
    print(dict(out[TARGET_VAR].attrs))

    # -------------------------
    # Final validation
    # -------------------------
    print_header("8) FINAL VALIDATION")
    # shapes
    assert out[TARGET_VAR].dims == ds_tp_sub[TARGET_VAR].dims, "Output dims do not match template dims"
    assert out[TARGET_VAR].shape == ds_tp_sub[TARGET_VAR].shape, "Output shape does not match template shape"
    print("Shape/dim check passed.")

    # coords exact equality
    for c in ["time", "plev", "lat", "lon"]:
        same = np.array_equal(out[c].values, ds_tp_sub[c].values)
        print(f"Coord equality check for {c}: {same}")
        assert same, f"Coordinate {c} does not exactly match template"

    # month alignment check
    print("\nMonth alignment check via SWOOSH year/month vs selected template months:")
    tmeta_sel = [tmeta_all[i] for i in keep_template_idx]
    for i, (_, _, _, y, m) in enumerate(tmeta_sel):
        sy = int(ds_sw_sub["year"].values[i])
        sm = int(ds_sw_sub["month"].values[i])
        ok = (y == sy and m == sm)
        print(f"  out[{i:2d}] template={y:04d}-{m:02d} ; source={sy:04d}-{sm:02d} ; ok={ok}")
        assert ok, f"Time mismatch at output index {i}"

    # final stats month by month
    final_month_stats = []
    print("\nFinal output validity month by month:")
    for i in range(out.sizes["time"]):
        y = tmeta_sel[i][3]
        m = tmeta_sel[i][4]
        s = da_stats(out[TARGET_VAR].isel(time=i), f"output {y:04d}-{m:02d} {TARGET_VAR}")
        s["year"] = int(y)
        s["month"] = int(m)
        final_month_stats.append(s)
    debug["final_month_stats"] = final_month_stats

    # overall stats compare with template
    print("\nOverall comparison with template (for sanity, not equality):")
    s_out = da_stats(out[TARGET_VAR], "FINAL OUTPUT overall")
    s_tpl = da_stats(ds_tp_sub[TARGET_VAR], "TEMPLATE overall")
    debug["overall_stats"] = {"output": s_out, "template": s_tpl}

    # -------------------------
    # Save
    # -------------------------
    print_header("9) SAVE OUTPUT")
    enc = {
        TARGET_VAR: {"zlib": True, "complevel": 4, "dtype": "float32"},
        "time": {"dtype": "float64"},
        "plev": {"dtype": "float32"},
        "lat": {"dtype": "float32"},
        "lon": {"dtype": "float32"},
    }
    for v in ["time_bnds", "plev_bnds", "lat_bnds", "lon_bnds"]:
        if v in out:
            enc[v] = {"zlib": True, "complevel": 4}

    out.to_netcdf(OUT_FILE, format="NETCDF4", encoding=enc)
    print(f"Saved netCDF: {OUT_FILE}")

    # JSON debug summary
    debug["output_file"] = OUT_FILE
    debug["source_file"] = SWOOSH_FILE
    debug["template_file"] = TEMPLATE_FILE
    debug["processing_choices"] = {
        "source_var": SOURCE_VAR,
        "target_var": TARGET_VAR,
        "unit_conversion": "ppmv_to_mole_mole-1_times_1e-6",
        "time_matching": "exact year-month matching to template",
        "horizontal_interp": "linear",
        "vertical_interp": "linear_in_log_pressure",
        "outside_vertical_range": "nearest_edge_extrapolation",
        "output_time_coords": "copied exactly from template",
        "output_bounds": "copied exactly from template",
    }
    with open(DEBUG_JSON, "w", encoding="utf-8") as f:
        json.dump(debug, f, indent=2, ensure_ascii=False)
    print(f"Saved debug JSON: {DEBUG_JSON}")

    print_header("DONE")
    print("Processing finished successfully.")
    print(f"NetCDF output : {OUT_FILE}")
    print(f"Debug summary : {DEBUG_JSON}")
    print("\nRecommended immediate checks on server:")
    print(f"  ncdump -h {OUT_FILE} | head -80")
    print(f"  ncl_filedump {OUT_FILE} | head -120")
    print(f"  ls -lh {OUT_FILE}")
    print("\nThen compare against template:")
    print(f"  ncdump -h {TEMPLATE_FILE} | head -80")

if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import math
import warnings
from datetime import datetime

import numpy as np
import xarray as xr

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================
# CONFIG
# =========================
SWOOSH_FILE = "/mnt/soclim0/public_data/weiji/swoosh/swoosh-v02.72-198401-202601-lonlatpress-20deg-5deg-L31.nc"
TEMPLATE_FILE = "/mnt/soclim0/andreas/vmro3_input4MIPs_ozone_CMIP6_UReading-CCMI_2020.nc"

SW_VAR = "combinedo3q"
OUT_VAR = "vmro3"

OUT_DIR = "/mnt/soclim0/public_data/weiji/swoosh/processed_like_input4MIPs"
OUT_NC = os.path.join(
    OUT_DIR,
    "vmro3_SWOOSHv02.72_combinedo3q_remapped_to_CMIP6_template_201912-202101.nc"
)

LOW_P_FILL_MODE = "nearest"
HIGH_P_FILL_MODE = "nearest"
USE_NEAREST_HORIZONTAL_FALLBACK = False


# =========================
# HELPERS
# =========================
def stats_np(arr):
    arr = np.asarray(arr)
    finite = np.isfinite(arr)
    n_total = arr.size
    n_finite = int(finite.sum())
    n_nan = int(np.isnan(arr).sum())
    n_neg = int(np.sum(finite & (arr < 0)))
    frac = n_finite / n_total if n_total > 0 else np.nan
    if n_finite > 0:
        vals = arr[finite]
        return {
            "shape": tuple(arr.shape),
            "n_finite": n_finite,
            "n_total": int(n_total),
            "n_nan": n_nan,
            "n_neg": n_neg,
            "frac": float(frac),
            "min": float(vals.min()),
            "max": float(vals.max()),
            "mean": float(vals.mean()),
        }
    return {
        "shape": tuple(arr.shape),
        "n_finite": 0,
        "n_total": int(n_total),
        "n_nan": n_nan,
        "n_neg": n_neg,
        "frac": float(frac),
        "min": None,
        "max": None,
        "mean": None,
    }


def print_stats(label, arr):
    s = stats_np(arr)
    print(
        f"{label}: shape={s['shape']}, "
        f"finite={s['n_finite']}/{s['n_total']}, "
        f"nan={s['n_nan']}, neg={s['n_neg']}, "
        f"frac={s['frac']:.6f}, min={s['min']}, max={s['max']}, mean={s['mean']}"
    )


def open_dataset_safely(path):
    engines = [None, "netcdf4", "h5netcdf", "scipy"]
    last_err = None
    for eng in engines:
        try:
            kwargs = dict(decode_times=False, mask_and_scale=True)
            if eng is None:
                return xr.open_dataset(path, **kwargs)
            return xr.open_dataset(path, engine=eng, **kwargs)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to open {path}. Last error: {last_err}")


def ym_from_month_index(month_index, base_year=1850, base_month=1):
    total = (base_year * 12 + (base_month - 1)) + int(month_index)
    year = total // 12
    month = total % 12 + 1
    return int(year), int(month)


def infer_template_year_months(template_time):
    out = []
    for i, t in enumerate(template_time):
        month_index = math.floor(float(t))
        y, m = ym_from_month_index(month_index, base_year=1850, base_month=1)
        out.append((i, y, m))
    return out


def add_cyclic_lon(da):
    lon = da["lon"].values.astype(float)
    left = da.isel(lon=-1).copy(deep=True).expand_dims(lon=[float(lon[-1] - 360.0)])
    right = da.isel(lon=0).copy(deep=True).expand_dims(lon=[float(lon[0] + 360.0)])
    return xr.concat([left, da, right], dim="lon").sortby("lon")


def add_lat_edges(da):
    lat = da["lat"].values.astype(float)
    pieces = []
    if lat[0] > -90.0:
        pieces.append(da.isel(lat=0).copy(deep=True).expand_dims(lat=[-90.0]))
    pieces.append(da)
    if lat[-1] < 90.0:
        pieces.append(da.isel(lat=-1).copy(deep=True).expand_dims(lat=[90.0]))
    return xr.concat(pieces, dim="lat").sortby("lat")


def interp_profile_logp(profile, src_p, dst_p, low_mode="nearest", high_mode="nearest"):
    prof = np.asarray(profile, dtype=np.float64)
    src_p = np.asarray(src_p, dtype=np.float64)
    dst_p = np.asarray(dst_p, dtype=np.float64)

    out = np.full(dst_p.shape, np.nan, dtype=np.float64)

    valid = np.isfinite(prof) & np.isfinite(src_p) & (src_p > 0)
    n_valid = int(valid.sum())
    if n_valid == 0:
        return out

    p_valid = src_p[valid]
    v_valid = prof[valid]

    order = np.argsort(p_valid)
    p_valid = p_valid[order]
    v_valid = v_valid[order]

    pmin = p_valid.min()
    pmax = p_valid.max()

    if n_valid == 1:
        out[:] = v_valid[0]
        return out

    inside = (dst_p >= pmin) & (dst_p <= pmax)
    if inside.any():
        out[inside] = np.interp(np.log(dst_p[inside]), np.log(p_valid), v_valid)

    low_out = dst_p > pmax
    if low_out.any() and low_mode == "nearest":
        out[low_out] = v_valid[-1]

    high_out = dst_p < pmin
    if high_out.any() and high_mode == "nearest":
        out[high_out] = v_valid[0]

    return out


# =========================
# MAIN BLOCK
# =========================
os.makedirs(OUT_DIR, exist_ok=True)

ds_sw = open_dataset_safely(SWOOSH_FILE)
ds_tp = open_dataset_safely(TEMPLATE_FILE)

# 1) exact month matching
tpl_matches = infer_template_year_months(ds_tp["time"].values)
sw_year = ds_sw["year"].values.astype(int)
sw_month = ds_sw["month"].values.astype(int)
sw_map = {(int(y), int(m)): i for i, (y, m) in enumerate(zip(sw_year, sw_month))}
sw_indices = [sw_map[(y, m)] for _, y, m in tpl_matches]

print("Matched template months:")
for (i, y, m), sw_idx in zip(tpl_matches, sw_indices):
    print(f"  tpl[{i:2d}] -> {y:04d}-{m:02d} -> SWOOSH idx {sw_idx}")

# 2) source field
src = ds_sw[SW_VAR].isel(time=sw_indices).astype(np.float64) * 1e-6
print_stats("Source first month after unit conversion", src.isel(time=0).values)

# 3) lon convert + prepare
src = src.assign_coords(lon=(src["lon"].values % 360.0)).sortby("lon")
src = src.rename(level="plev")
src = add_cyclic_lon(src)
src = add_lat_edges(src)

# 4) horizontal interpolation
src_lin = src.interp(
    lat=ds_tp["lat"],
    lon=ds_tp["lon"],
    method="linear",
    kwargs={"fill_value": np.nan}
)

if USE_NEAREST_HORIZONTAL_FALLBACK:
    src_nn = src.interp(
        lat=ds_tp["lat"],
        lon=ds_tp["lon"],
        method="nearest",
        kwargs={"fill_value": np.nan}
    )
    src_horiz = src_lin.where(np.isfinite(src_lin), src_nn)
else:
    src_horiz = src_lin

src_horiz = src_horiz.transpose("time", "plev", "lat", "lon")
print_stats("After horizontal interpolation, first month", src_horiz.isel(time=0).values)

# 5) vertical interpolation
src_plev = src_horiz["plev"].values.astype(np.float64)
dst_plev = ds_tp["plev"].values.astype(np.float64)

src_prof = src_horiz.transpose("time", "lat", "lon", "plev")

vert = xr.apply_ufunc(
    interp_profile_logp,
    src_prof,
    xr.DataArray(src_plev, dims=["plev"]),
    xr.DataArray(dst_plev, dims=["plev_out"]),
    input_core_dims=[["plev"], ["plev"], ["plev_out"]],
    output_core_dims=[["plev_out"]],
    vectorize=True,
    dask="forbidden",
    output_dtypes=[np.float64],
    kwargs={"low_mode": LOW_P_FILL_MODE, "high_mode": HIGH_P_FILL_MODE},
    dask_gufunc_kwargs={"output_sizes": {"plev_out": len(dst_plev)}},
)

out_var = (
    vert.rename({"plev_out": "plev"})
    .assign_coords(
        time=ds_tp["time"].values,
        plev=ds_tp["plev"].values,
        lat=ds_tp["lat"].values,
        lon=ds_tp["lon"].values,
    )
    .transpose("time", "plev", "lat", "lon")
)

print_stats("After vertical interpolation, first month", out_var.isel(time=0).values)

# 6) assemble output dataset
out_ds = xr.Dataset(
    coords={
        "time": ds_tp["time"],
        "plev": ds_tp["plev"],
        "lat": ds_tp["lat"],
        "lon": ds_tp["lon"],
    }
)

for bname in ["time_bnds", "plev_bnds", "lat_bnds", "lon_bnds"]:
    if bname in ds_tp:
        out_ds[bname] = ds_tp[bname]

out_ds[OUT_VAR] = out_var.astype(np.float32)
out_ds[OUT_VAR].attrs = {
    "long_name": "vmro3",
    "standard_name": "mole_fraction_of_ozone_in_air",
    "units": "mole mole-1",
    "cell_methods": "time:mean",
}

# 7) correct global metadata
out_ds.attrs = {
    "Conventions": "CF-1.6",
    "title": "SWOOSH ozone remapped to CMIP6-style vmro3 template grid",
    "source": "SWOOSH v02.72 combinedo3q",
    "comment": (
        "Monthly mean ozone field constructed from SWOOSH v02.72 variable "
        "'combinedo3q' and remapped onto the grid, pressure levels, and time axis "
        "of the reference CMIP6-style vmro3 template file. Units converted from "
        "ppmv to mole mole-1. Horizontal interpolation performed on the lon-lat "
        "grid; vertical interpolation performed in log-pressure space. Missing "
        "values and negative values were preserved rather than artificially filled "
        "or clipped."
    ),
    "institution": "Derived from NOAA SWOOSH data and remapped by Weiji Hu",
    "created_by": "Weiji Hu",
    "creation_date": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
    "variable_id": "vmro3",
    "dataset_category": "ozone",
    "frequency": "mon",
    "grid": "Template lat-lon-pressure grid based on vmro3_input4MIPs_ozone_CMIP6_UReading-CCMI_2020.nc",
    "history": (
        "Created from SWOOSH v02.72 combinedo3q; "
        "exact template month matching (2019-12 to 2021-01); "
        "unit conversion ppmv -> mole mole-1; "
        "horizontal interpolation to template lat/lon; "
        "log-pressure interpolation to template plev; "
        "no artificial filling or clipping applied."
    ),
}

# 8) analysis / comparison
print("\n=== FINAL COMPARISON WITH TEMPLATE ===")
print_stats("OUTPUT overall", out_ds[OUT_VAR].values)
print_stats("TEMPLATE overall", ds_tp[OUT_VAR].values)

print("\nCoordinate equality:")
for c in ["time", "plev", "lat", "lon"]:
    same = np.allclose(out_ds[c].values, ds_tp[c].values, equal_nan=True)
    print(f"  {c}: {same}")

print("\nBounds equality:")
for b in ["time_bnds", "plev_bnds", "lat_bnds", "lon_bnds"]:
    if b in out_ds and b in ds_tp:
        same = np.allclose(out_ds[b].values, ds_tp[b].values, equal_nan=True)
        print(f"  {b}: {same}")

print("\nMonth-by-month diagnostics:")
for i, (_, y, m) in enumerate(tpl_matches):
    print_stats(f"{y:04d}-{m:02d} OUTPUT", out_ds[OUT_VAR].isel(time=i).values)

# 9) save
encoding = {
    OUT_VAR: {"zlib": True, "complevel": 4, "dtype": "float32"},
    "time": {"dtype": "float64"},
    "plev": {"dtype": "float32"},
    "lat": {"dtype": "float32"},
    "lon": {"dtype": "float32"},
}
out_ds.to_netcdf(OUT_NC, format="NETCDF4", encoding=encoding)

print(f"\nSaved: {OUT_NC}")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import xarray as xr
import netCDF4 as nc
import numpy as np

REF = "/mnt/soclim0/andreas/vmro3_input4MIPs_ozone_CMIP6_UReading-CCMI_2020.nc"
NEW = "/mnt/soclim0/public_data/weiji/swoosh/processed_like_input4MIPs/vmro3_SWOOSHv02.72_combinedo3q_remapped_to_CMIP6_template_201912-202101.nc"

def print_header(title):
    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)

def safe_attr_dict(obj):
    out = {}
    for k in obj.ncattrs():
        try:
            out[k] = obj.getncattr(k)
        except Exception as e:
            out[k] = f"<ERROR: {e}>"
    return out

def compare_sets(name, a, b):
    only_a = sorted(set(a) - set(b))
    only_b = sorted(set(b) - set(a))
    common = sorted(set(a) & set(b))
    print(f"{name}:")
    print(f"  common   ({len(common)}): {common}")
    print(f"  only REF ({len(only_a)}): {only_a}")
    print(f"  only NEW ({len(only_b)}): {only_b}")
    return common, only_a, only_b

def compare_attr_dicts(name, a, b):
    print(f"\n{name}")
    keys = sorted(set(a) | set(b))
    for k in keys:
        va = a.get(k, "<MISSING>")
        vb = b.get(k, "<MISSING>")
        same = False
        try:
            if isinstance(va, np.ndarray) or isinstance(vb, np.ndarray):
                same = np.array_equal(np.asarray(va), np.asarray(vb))
            else:
                same = (va == vb)
        except Exception:
            same = False
        flag = "OK" if same else "DIFF"
        print(f"  [{flag}] {k}")
        if not same:
            print(f"      REF: {va}")
            print(f"      NEW: {vb}")

def var_summary(ds, varname):
    v = ds.variables[varname]
    out = {
        "dtype": str(v.dtype),
        "dims": v.dimensions,
        "shape": v.shape,
        "attrs": safe_attr_dict(v),
    }
    # netCDF4 low-level chunk/compression info
    try:
        out["chunking"] = v.chunking()
    except Exception as e:
        out["chunking"] = f"<ERROR: {e}>"
    try:
        out["filters"] = v.filters()
    except Exception as e:
        out["filters"] = f"<ERROR: {e}>"
    return out

def main():
    ref = nc.Dataset(REF, "r")
    new = nc.Dataset(NEW, "r")

    print_header("GLOBAL DIMENSIONS")
    ref_dims = {k: len(v) for k, v in ref.dimensions.items()}
    new_dims = {k: len(v) for k, v in new.dimensions.items()}
    compare_attr_dicts("Dimensions comparison", ref_dims, new_dims)

    print_header("GLOBAL ATTRIBUTES")
    compare_attr_dicts("Global attrs", safe_attr_dict(ref), safe_attr_dict(new))

    print_header("VARIABLE LIST")
    common, only_ref, only_new = compare_sets("Variables", ref.variables.keys(), new.variables.keys())

    print_header("PER-VARIABLE COMPARISON")
    for vname in common:
        print("\n" + "-" * 120)
        print(f"Variable: {vname}")
        r = var_summary(ref, vname)
        n = var_summary(new, vname)

        print(f"  dtype     REF={r['dtype']} | NEW={n['dtype']}")
        print(f"  dims      REF={r['dims']} | NEW={n['dims']}")
        print(f"  shape     REF={r['shape']} | NEW={n['shape']}")
        print(f"  chunking  REF={r['chunking']} | NEW={n['chunking']}")
        print(f"  filters   REF={r['filters']} | NEW={n['filters']}")

        compare_attr_dicts(f"  attrs for {vname}", r["attrs"], n["attrs"])

    ref.close()
    new.close()

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import math
import warnings
import numpy as np
import xarray as xr

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# CONFIG
# =============================================================================
SWOOSH_FILE = "/mnt/soclim0/public_data/weiji/swoosh/swoosh-v02.72-198401-202601-lonlatpress-20deg-5deg-L31.nc"
TEMPLATE_FILE = "/mnt/soclim0/andreas/vmro3_input4MIPs_ozone_CMIP6_UReading-CCMI_2020.nc"

SW_VAR = "combinedo3q"

LOW_P_FILL_MODE = "nearest"
HIGH_P_FILL_MODE = "nearest"


# =============================================================================
# HELPERS
# =============================================================================
def print_header(title: str):
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)


def open_dataset_safely(path: str):
    engines = [None, "netcdf4", "h5netcdf", "scipy"]
    last_err = None
    for eng in engines:
        try:
            kwargs = dict(decode_times=False, mask_and_scale=True)
            if eng is None:
                ds = xr.open_dataset(path, **kwargs)
                return ds, "default"
            else:
                ds = xr.open_dataset(path, engine=eng, **kwargs)
                return ds, eng
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to open {path}. Last error: {last_err}")


def ym_from_month_index(month_index: int, base_year: int = 1850, base_month: int = 1):
    total = (base_year * 12 + (base_month - 1)) + int(month_index)
    year = total // 12
    month = total % 12 + 1
    return int(year), int(month)


def infer_template_year_months(template_time: np.ndarray):
    out = []
    for i, t in enumerate(template_time):
        month_index = math.floor(float(t))
        y, m = ym_from_month_index(month_index, base_year=1850, base_month=1)
        out.append((i, float(t), month_index, y, m))
    return out


def add_cyclic_lon(da: xr.DataArray) -> xr.DataArray:
    lon = da["lon"].values.astype(float)
    if not np.all(np.diff(lon) > 0):
        raise ValueError("Longitude must be strictly increasing before cyclic padding.")

    left = da.isel(lon=-1).copy(deep=True).expand_dims(lon=[float(lon[-1] - 360.0)])
    right = da.isel(lon=0).copy(deep=True).expand_dims(lon=[float(lon[0] + 360.0)])

    out = xr.concat([left, da, right], dim="lon").sortby("lon")
    return out


def add_lat_edges(da: xr.DataArray) -> xr.DataArray:
    lat = da["lat"].values.astype(float)
    if not np.all(np.diff(lat) > 0):
        raise ValueError("Latitude must be strictly increasing before edge padding.")

    pieces = []

    if lat[0] > -90.0:
        south = da.isel(lat=0).copy(deep=True).expand_dims(lat=[-90.0])
        pieces.append(south)

    pieces.append(da)

    if lat[-1] < 90.0:
        north = da.isel(lat=-1).copy(deep=True).expand_dims(lat=[90.0])
        pieces.append(north)

    out = xr.concat(pieces, dim="lat").sortby("lat")
    return out


def interp_profile_logp(profile, src_p, dst_p, low_mode="nearest", high_mode="nearest"):
    prof = np.asarray(profile, dtype=np.float64)
    src_p = np.asarray(src_p, dtype=np.float64)
    dst_p = np.asarray(dst_p, dtype=np.float64)

    out = np.full(dst_p.shape, np.nan, dtype=np.float64)

    valid = np.isfinite(prof) & np.isfinite(src_p) & (src_p > 0)
    n_valid = int(valid.sum())
    if n_valid == 0:
        return out

    p_valid = src_p[valid]
    v_valid = prof[valid]

    order = np.argsort(p_valid)
    p_valid = p_valid[order]
    v_valid = v_valid[order]

    pmin = p_valid.min()
    pmax = p_valid.max()

    if n_valid == 1:
        out[:] = v_valid[0]
        return out

    inside = (dst_p >= pmin) & (dst_p <= pmax)
    if inside.any():
        out[inside] = np.interp(
            np.log(dst_p[inside]),
            np.log(p_valid),
            v_valid
        )

    low_out = dst_p > pmax
    if low_out.any():
        if low_mode == "nearest":
            out[low_out] = v_valid[-1]
        else:
            raise ValueError(f"Unknown LOW_P_FILL_MODE: {low_mode}")

    high_out = dst_p < pmin
    if high_out.any():
        if high_mode == "nearest":
            out[high_out] = v_valid[0]
        else:
            raise ValueError(f"Unknown HIGH_P_FILL_MODE: {high_mode}")

    return out


def summarize_column_nan_mask(mask_2d, lat_vals, lon_vals, label):
    """
    mask_2d: (lat, lon), True means this whole vertical column is NaN
    """
    n_lat, n_lon = mask_2d.shape
    total = n_lat * n_lon
    count = int(mask_2d.sum())
    print(f"{label}: all-NaN columns = {count}/{total}")

    by_lat = mask_2d.sum(axis=1)
    bad_lat_idx = np.where(by_lat > 0)[0]

    print("Rows with at least one all-NaN column:")
    for j in bad_lat_idx:
        print(
            f"  lat_idx={j:3d}, lat={lat_vals[j]:8.3f}, "
            f"bad_lon_count={int(by_lat[j])}/{n_lon}"
        )

    full_bad_lat_idx = np.where(by_lat == n_lon)[0]
    print("Rows where ALL longitudes are all-NaN:")
    if len(full_bad_lat_idx) == 0:
        print("  none")
    else:
        for j in full_bad_lat_idx:
            print(f"  lat_idx={j:3d}, lat={lat_vals[j]:8.3f}")

    by_lon = mask_2d.sum(axis=0)
    full_bad_lon_idx = np.where(by_lon == n_lat)[0]
    print("Columns where ALL latitudes are all-NaN:")
    if len(full_bad_lon_idx) == 0:
        print("  none")
    else:
        for i in full_bad_lon_idx:
            print(f"  lon_idx={i:3d}, lon={lon_vals[i]:8.3f}")


def print_small_mask(mask_2d, lat_vals, lon_vals, max_rows=20):
    """
    简易可视化：用 1/0 看哪些纬向带坏了
    """
    print("\nCompact lat-row summary (1 means all 144 lon columns are NaN for that lat row):")
    by_lat_full = (mask_2d.sum(axis=1) == mask_2d.shape[1]).astype(int)
    rows = []
    for j, flag in enumerate(by_lat_full):
        rows.append(f"{j:02d}:{lat_vals[j]:6.1f}:{flag}")
    for k in range(0, len(rows), max_rows):
        print("  " + " | ".join(rows[k:k+max_rows]))


# =============================================================================
# MAIN
# =============================================================================
def main():
    print_header("1) OPEN DATASETS")
    ds_sw, eng_sw = open_dataset_safely(SWOOSH_FILE)
    ds_tp, eng_tp = open_dataset_safely(TEMPLATE_FILE)
    print(f"Opened SWOOSH with engine   : {eng_sw}")
    print(f"Opened TEMPLATE with engine : {eng_tp}")

    print_header("2) TIME MATCHING")
    tpl_time = ds_tp["time"].values
    tpl_matches = infer_template_year_months(tpl_time)

    sw_year = ds_sw["year"].values.astype(int)
    sw_month = ds_sw["month"].values.astype(int)
    sw_map = {(int(y), int(m)): i for i, (y, m) in enumerate(zip(sw_year, sw_month))}

    matched = []
    sw_indices = []
    for tpl_idx, raw_t, month_index, y, m in tpl_matches:
        sw_idx = sw_map[(y, m)]
        matched.append((tpl_idx, raw_t, month_index, y, m, sw_idx))
        sw_indices.append(sw_idx)
        print(f"tpl_idx={tpl_idx:2d}, ym={y:04d}-{m:02d}, sw_idx={sw_idx}")

    # ----------------------------------------------------------------------------
    # build source
    # ----------------------------------------------------------------------------
    print_header("3) BUILD SOURCE")
    src = ds_sw[SW_VAR].isel(time=sw_indices).astype(np.float64) * 1e-6
    src = src.assign_coords(lon=(src["lon"].values % 360.0)).sortby("lon")
    src = src.rename(level="plev")

    print(f"src dims: {src.dims}, shape={src.shape}")
    print(f"src lat range: {float(src['lat'].min())} .. {float(src['lat'].max())}")
    print(f"src lon range: {float(src['lon'].min())} .. {float(src['lon'].max())}")
    print(f"src plev range: {float(src['plev'].min())} .. {float(src['plev'].max())}")

    # ----------------------------------------------------------------------------
    # source all-NaN vertical columns
    # ----------------------------------------------------------------------------
    print_header("4) SOURCE ALL-NaN VERTICAL COLUMNS")
    # source dims: (time, plev, lat, lon)
    src_col_allnan = src.isnull().all(dim="plev")  # (time, lat, lon)

    counts = src_col_allnan.sum(dim=("lat", "lon")).values.astype(int)
    print("Per month source all-NaN column counts (lat-lon columns where all 31 source levels are NaN):")
    for i, (_, _, _, y, m, _) in enumerate(matched):
        print(f"  {y:04d}-{m:02d}: {counts[i]}")

    t0 = 0
    y0 = matched[t0][3]
    m0 = matched[t0][4]
    print(f"\nDetailed source-column diagnosis for first month: {y0:04d}-{m0:02d}")
    summarize_column_nan_mask(
        src_col_allnan.isel(time=t0).values,
        src["lat"].values,
        src["lon"].values,
        label="SOURCE"
    )

    # inspect source polar rows
    print("\nSource polar-row all-NaN-column counts by latitude:")
    by_lat_source = src_col_allnan.isel(time=t0).sum(dim="lon").values.astype(int)
    for j, latv in enumerate(src["lat"].values):
        if abs(latv) >= 70:
            print(f"  lat_idx={j:2d}, lat={latv:7.2f}, bad_lon_count={by_lat_source[j]}/{src.sizes['lon']}")

    # ----------------------------------------------------------------------------
    # horizontal interpolation
    # ----------------------------------------------------------------------------
    print_header("5) HORIZONTAL INTERPOLATION")
    src_h = add_cyclic_lon(src)
    src_h = add_lat_edges(src_h)

    target_lat = ds_tp["lat"]
    target_lon = ds_tp["lon"]

    src_lin = src_h.interp(
        lat=target_lat,
        lon=target_lon,
        method="linear",
        kwargs={"fill_value": np.nan}
    )

    print(f"After horiz interp dims: {src_lin.dims}, shape={src_lin.shape}")

    # standardize order to (time, plev, lat, lon)
    src_lin = src_lin.transpose("time", "plev", "lat", "lon")

    # horizontal all-NaN columns
    horiz_col_allnan = src_lin.isnull().all(dim="plev")  # (time, lat, lon)
    horiz_counts = horiz_col_allnan.sum(dim=("lat", "lon")).values.astype(int)

    print("\nPer month HORIZONTAL all-NaN column counts (after interpolation):")
    for i, (_, _, _, y, m, _) in enumerate(matched):
        print(f"  {y:04d}-{m:02d}: {horiz_counts[i]}")

    print(f"\nDetailed horizontal-column diagnosis for first month: {y0:04d}-{m0:02d}")
    summarize_column_nan_mask(
        horiz_col_allnan.isel(time=t0).values,
        target_lat.values,
        target_lon.values,
        label="HORIZONTAL"
    )
    print_small_mask(
        horiz_col_allnan.isel(time=t0).values,
        target_lat.values,
        target_lon.values
    )

    # compare source->horizontal increase
    print("\nChange in all-NaN column counts from SOURCE to HORIZONTAL:")
    for i, (_, _, _, y, m, _) in enumerate(matched):
        print(f"  {y:04d}-{m:02d}: source={counts[i]}, horizontal={horiz_counts[i]}, delta={horiz_counts[i]-counts[i]}")

    # ----------------------------------------------------------------------------
    # vertical interpolation
    # ----------------------------------------------------------------------------
    print_header("6) VERTICAL INTERPOLATION")
    src_plev = src_lin["plev"].values.astype(np.float64)
    dst_plev = ds_tp["plev"].values.astype(np.float64)

    src_prof = src_lin.transpose("time", "lat", "lon", "plev")

    vert = xr.apply_ufunc(
        interp_profile_logp,
        src_prof,
        xr.DataArray(src_plev, dims=["plev"]),
        xr.DataArray(dst_plev, dims=["plev_out"]),
        input_core_dims=[["plev"], ["plev"], ["plev_out"]],
        output_core_dims=[["plev_out"]],
        vectorize=True,
        dask="forbidden",
        output_dtypes=[np.float64],
        kwargs={"low_mode": LOW_P_FILL_MODE, "high_mode": HIGH_P_FILL_MODE},
        dask_gufunc_kwargs={"output_sizes": {"plev_out": len(dst_plev)}},
    )

    vert = vert.rename({"plev_out": "plev"}).assign_coords(plev=ds_tp["plev"])
    vert = vert.transpose("time", "plev", "lat", "lon")

    final_col_allnan = vert.isnull().all(dim="plev")  # (time, lat, lon)
    final_counts = final_col_allnan.sum(dim=("lat", "lon")).values.astype(int)

    print("Per month FINAL all-NaN column counts (after vertical interpolation):")
    for i, (_, _, _, y, m, _) in enumerate(matched):
        print(f"  {y:04d}-{m:02d}: {final_counts[i]}")

    print(f"\nDetailed final-column diagnosis for first month: {y0:04d}-{m0:02d}")
    summarize_column_nan_mask(
        final_col_allnan.isel(time=t0).values,
        target_lat.values,
        target_lon.values,
        label="FINAL"
    )
    print_small_mask(
        final_col_allnan.isel(time=t0).values,
        target_lat.values,
        target_lon.values
    )

    # ----------------------------------------------------------------------------
    # confirm whether final NaNs come from all-NaN horizontal columns
    # ----------------------------------------------------------------------------
    print_header("7) CONFIRMATION OF NaN ORIGIN")
    same_mask = (horiz_col_allnan.values == final_col_allnan.values)
    print(f"Horizontal all-NaN column mask == Final all-NaN column mask ? {bool(np.all(same_mask))}")

    for i, (_, _, _, y, m, _) in enumerate(matched):
        eq = bool(np.all(horiz_col_allnan.isel(time=i).values == final_col_allnan.isel(time=i).values))
        print(f"  {y:04d}-{m:02d}: identical={eq}")

    print("\nIf these are identical, it means:")
    print("- vertical interpolation is NOT creating new NaN columns")
    print("- it is simply preserving columns that were already all-NaN after horizontal interpolation")

    # ----------------------------------------------------------------------------
    # extra: inspect target lat rows responsible for repeated 76032 NaNs
    # ----------------------------------------------------------------------------
    print_header("8) TARGET LAT-ROW CONTRIBUTION TO FINAL NaNs")
    # count all-NaN vertical columns by lat row for first month
    row_bad_counts = final_col_allnan.isel(time=0).sum(dim="lon").values.astype(int)
    print("First month: bad all-NaN column count by target latitude row:")
    for j, latv in enumerate(target_lat.values):
        if row_bad_counts[j] > 0:
            print(f"  lat_idx={j:3d}, lat={latv:8.3f}, bad_lon_count={row_bad_counts[j]}/{target_lon.size}")

    total_bad_columns = int(final_col_allnan.isel(time=0).sum().item())
    total_bad_values = total_bad_columns * ds_tp["plev"].size
    print(f"\nFirst month bad lat-lon columns = {total_bad_columns}")
    print(f"If every bad column stays NaN through all 66 target levels, total NaNs = {total_bad_columns} * 66 = {total_bad_values}")

    # ----------------------------------------------------------------------------
    # extra: source polar coverage detail
    # ----------------------------------------------------------------------------
    print_header("9) SOURCE POLAR COVERAGE DETAIL (FIRST MONTH)")
    src0 = src.isel(time=0)
    for lat_pick in [-87.5, -82.5, 82.5, 87.5]:
        if lat_pick in src0["lat"].values:
            da = src0.sel(lat=lat_pick)
            # columns all-NaN over plev
            bad_lon = da.isnull().all(dim="plev").values
            print(f"Source lat={lat_pick:6.1f}: all-NaN vertical columns across lon = {int(bad_lon.sum())}/{src0.sizes['lon']}")

    print_header("DONE")
    print("Read the sections in this order:")
    print("1) SOURCE ALL-NaN VERTICAL COLUMNS")
    print("2) HORIZONTAL INTERPOLATION")
    print("3) FINAL all-NaN column counts")
    print("4) CONFIRMATION OF NaN ORIGIN")
    print("5) TARGET LAT-ROW CONTRIBUTION TO FINAL NaNs")

main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import math
import warnings
import numpy as np
import xarray as xr

warnings.filterwarnings("ignore", category=RuntimeWarning)

SWOOSH_FILE = "/mnt/soclim0/public_data/weiji/swoosh/swoosh-v02.72-198401-202601-lonlatpress-20deg-5deg-L31.nc"
TEMPLATE_FILE = "/mnt/soclim0/andreas/vmro3_input4MIPs_ozone_CMIP6_UReading-CCMI_2020.nc"

CANDIDATES = [
    "combinedo3q",
    "combinedeqfillo3q",
    "combinedanomfillo3q",
    "combinedeqfillanomfillo3q",
]

LOW_P_FILL_MODE = "nearest"
HIGH_P_FILL_MODE = "nearest"


def print_header(title: str):
    print("\n" + "=" * 120)
    print(title)
    print("=" * 120)


def open_dataset_safely(path: str):
    engines = [None, "netcdf4", "h5netcdf", "scipy"]
    last_err = None
    for eng in engines:
        try:
            kwargs = dict(decode_times=False, mask_and_scale=True)
            if eng is None:
                ds = xr.open_dataset(path, **kwargs)
                return ds, "default"
            else:
                ds = xr.open_dataset(path, engine=eng, **kwargs)
                return ds, eng
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Failed to open {path}. Last error: {last_err}")


def ym_from_month_index(month_index: int, base_year: int = 1850, base_month: int = 1):
    total = (base_year * 12 + (base_month - 1)) + int(month_index)
    year = total // 12
    month = total % 12 + 1
    return int(year), int(month)


def infer_template_year_months(template_time: np.ndarray):
    out = []
    for i, t in enumerate(template_time):
        month_index = math.floor(float(t))
        y, m = ym_from_month_index(month_index, base_year=1850, base_month=1)
        out.append((i, float(t), month_index, y, m))
    return out


def stats_np(arr: np.ndarray):
    arr = np.asarray(arr)
    finite = np.isfinite(arr)
    n_total = arr.size
    n_finite = int(finite.sum())
    n_nan = int(np.isnan(arr).sum())
    n_neg = int(np.sum(finite & (arr < 0)))
    frac = n_finite / n_total if n_total > 0 else np.nan

    if n_finite > 0:
        vals = arr[finite]
        return {
            "shape": tuple(arr.shape),
            "n_finite": n_finite,
            "n_total": int(n_total),
            "n_nan": n_nan,
            "n_neg": n_neg,
            "frac": float(frac),
            "min": float(vals.min()),
            "max": float(vals.max()),
            "mean": float(vals.mean()),
        }
    else:
        return {
            "shape": tuple(arr.shape),
            "n_finite": 0,
            "n_total": int(n_total),
            "n_nan": n_nan,
            "n_neg": n_neg,
            "frac": float(frac),
            "min": None,
            "max": None,
            "mean": None,
        }


def add_cyclic_lon(da: xr.DataArray) -> xr.DataArray:
    lon = da["lon"].values.astype(float)
    left = da.isel(lon=-1).copy(deep=True).expand_dims(lon=[float(lon[-1] - 360.0)])
    right = da.isel(lon=0).copy(deep=True).expand_dims(lon=[float(lon[0] + 360.0)])
    return xr.concat([left, da, right], dim="lon").sortby("lon")


def add_lat_edges(da: xr.DataArray) -> xr.DataArray:
    lat = da["lat"].values.astype(float)
    pieces = []
    if lat[0] > -90.0:
        south = da.isel(lat=0).copy(deep=True).expand_dims(lat=[-90.0])
        pieces.append(south)
    pieces.append(da)
    if lat[-1] < 90.0:
        north = da.isel(lat=-1).copy(deep=True).expand_dims(lat=[90.0])
        pieces.append(north)
    return xr.concat(pieces, dim="lat").sortby("lat")


def interp_profile_logp(profile, src_p, dst_p, low_mode="nearest", high_mode="nearest"):
    prof = np.asarray(profile, dtype=np.float64)
    src_p = np.asarray(src_p, dtype=np.float64)
    dst_p = np.asarray(dst_p, dtype=np.float64)

    out = np.full(dst_p.shape, np.nan, dtype=np.float64)

    valid = np.isfinite(prof) & np.isfinite(src_p) & (src_p > 0)
    n_valid = int(valid.sum())
    if n_valid == 0:
        return out

    p_valid = src_p[valid]
    v_valid = prof[valid]

    order = np.argsort(p_valid)
    p_valid = p_valid[order]
    v_valid = v_valid[order]

    pmin = p_valid.min()
    pmax = p_valid.max()

    if n_valid == 1:
        out[:] = v_valid[0]
        return out

    inside = (dst_p >= pmin) & (dst_p <= pmax)
    if inside.any():
        out[inside] = np.interp(np.log(dst_p[inside]), np.log(p_valid), v_valid)

    low_out = dst_p > pmax
    if low_out.any():
        if low_mode == "nearest":
            out[low_out] = v_valid[-1]

    high_out = dst_p < pmin
    if high_out.any():
        if high_mode == "nearest":
            out[high_out] = v_valid[0]

    return out


def remap_candidate_to_template(da_src, ds_tp):
    """
    da_src input dims: (time, level, lat, lon)
    returns remapped DataArray dims: (time, plev, lat, lon)
    """
    src = da_src.astype(np.float64) * 1e-6
    src = src.assign_coords(lon=(src["lon"].values % 360.0)).sortby("lon")
    src = src.rename(level="plev")

    src_h = add_cyclic_lon(src)
    src_h = add_lat_edges(src_h)

    src_lin = src_h.interp(
        lat=ds_tp["lat"],
        lon=ds_tp["lon"],
        method="linear",
        kwargs={"fill_value": np.nan}
    )
    src_lin = src_lin.transpose("time", "plev", "lat", "lon")

    src_plev = src_lin["plev"].values.astype(np.float64)
    dst_plev = ds_tp["plev"].values.astype(np.float64)

    src_prof = src_lin.transpose("time", "lat", "lon", "plev")

    vert = xr.apply_ufunc(
        interp_profile_logp,
        src_prof,
        xr.DataArray(src_plev, dims=["plev"]),
        xr.DataArray(dst_plev, dims=["plev_out"]),
        input_core_dims=[["plev"], ["plev"], ["plev_out"]],
        output_core_dims=[["plev_out"]],
        vectorize=True,
        dask="forbidden",
        output_dtypes=[np.float64],
        kwargs={"low_mode": LOW_P_FILL_MODE, "high_mode": HIGH_P_FILL_MODE},
        dask_gufunc_kwargs={"output_sizes": {"plev_out": len(dst_plev)}},
    )

    vert = vert.rename({"plev_out": "plev"}).assign_coords(plev=ds_tp["plev"])
    vert = vert.transpose("time", "plev", "lat", "lon")
    return vert


def main():
    print_header("OPEN DATASETS")
    ds_sw, eng_sw = open_dataset_safely(SWOOSH_FILE)
    ds_tp, eng_tp = open_dataset_safely(TEMPLATE_FILE)
    print(f"Opened SWOOSH with engine   : {eng_sw}")
    print(f"Opened TEMPLATE with engine : {eng_tp}")

    print_header("TARGET MONTHS")
    tpl_matches = infer_template_year_months(ds_tp["time"].values)
    sw_year = ds_sw["year"].values.astype(int)
    sw_month = ds_sw["month"].values.astype(int)
    sw_map = {(int(y), int(m)): i for i, (y, m) in enumerate(zip(sw_year, sw_month))}

    matched = []
    sw_indices = []
    for tpl_idx, raw_t, month_index, y, m in tpl_matches:
        sw_idx = sw_map[(y, m)]
        matched.append((tpl_idx, y, m, sw_idx))
        sw_indices.append(sw_idx)
        print(f"tpl_idx={tpl_idx:2d}, ym={y:04d}-{m:02d}, sw_idx={sw_idx}")

    print_header("AVAILABLE combined*o3q VARIABLES IN THIS FILE")
    all_vars = list(ds_sw.data_vars)
    combined_candidates_in_file = [v for v in all_vars if v.startswith("combined") and "o3q" in v]
    for v in combined_candidates_in_file:
        print(v)

    print_header("COMPARE CANDIDATES")
    for var in CANDIDATES:
        print_header(f"CANDIDATE: {var}")

        if var not in ds_sw:
            print("NOT FOUND in this SWOOSH file.")
            continue

        da = ds_sw[var].isel(time=sw_indices)

        print("SOURCE monthly summary:")
        for i, (_, y, m, _) in enumerate(matched):
            s = stats_np(da.isel(time=i).values)
            print(
                f"  {y:04d}-{m:02d} : "
                f"finite={s['n_finite']}/{s['n_total']} "
                f"frac={s['frac']:.6f} nan={s['n_nan']} neg={s['n_neg']}"
            )

        source_col_allnan = da.isnull().all(dim="level")
        source_counts = source_col_allnan.sum(dim=("lat", "lon")).values.astype(int)
        print("SOURCE all-NaN vertical columns per month:", source_counts.tolist())

        first = source_col_allnan.isel(time=0)
        by_lat = first.sum(dim="lon").values.astype(int)
        bad_rows = np.where(by_lat > 0)[0]
        print("SOURCE first-month bad latitude rows:")
        if len(bad_rows) == 0:
            print("  none")
        else:
            for j in bad_rows:
                print(
                    f"  lat_idx={j:2d}, lat={float(ds_sw['lat'].values[j]):7.2f}, "
                    f"bad_lon={by_lat[j]}/{ds_sw.sizes['lon']}"
                )

        # remap
        remapped = remap_candidate_to_template(da, ds_tp)

        print("\nREMAPPED monthly summary:")
        for i, (_, y, m, _) in enumerate(matched):
            s = stats_np(remapped.isel(time=i).values)
            print(
                f"  {y:04d}-{m:02d} : "
                f"finite={s['n_finite']}/{s['n_total']} "
                f"frac={s['frac']:.6f} nan={s['n_nan']} neg={s['n_neg']}"
            )

        final_col_allnan = remapped.isnull().all(dim="plev")
        final_counts = final_col_allnan.sum(dim=("lat", "lon")).values.astype(int)
        print("REMAPPED all-NaN vertical columns per month:", final_counts.tolist())

        first_f = final_col_allnan.isel(time=0)
        by_lat_f = first_f.sum(dim="lon").values.astype(int)
        bad_rows_f = np.where(by_lat_f > 0)[0]
        print("REMAPPED first-month bad latitude rows:")
        if len(bad_rows_f) == 0:
            print("  none")
        else:
            for j in bad_rows_f:
                print(
                    f"  lat_idx={j:2d}, lat={float(ds_tp['lat'].values[j]):8.3f}, "
                    f"bad_lon={by_lat_f[j]}/{ds_tp.sizes['lon']}"
                )

        overall = stats_np(remapped.values)
        print("\nREMAPPED overall:")
        print(
            f"  finite={overall['n_finite']}/{overall['n_total']} "
            f"frac={overall['frac']:.6f} nan={overall['n_nan']} neg={overall['n_neg']} "
            f"min={overall['min']} max={overall['max']}"
        )


if __name__ == "__main__":
    main()

In [ ]:
import xarray as xr

fn = "/mnt/soclim0/public_data/weiji/swoosh/swoosh-v02.72-198401-202601-lonlatpress-20deg-5deg-L31.nc"
ds = xr.open_dataset(fn, decode_times=False)

vars_all = list(ds.data_vars)

print("=== all variables containing 'fill' ===")
fill_vars = [v for v in vars_all if "fill" in v.lower()]
for v in fill_vars:
    print(v)

print("\n=== O3 variables containing 'fill' ===")
fill_o3_vars = [v for v in vars_all if ("o3" in v.lower() and "fill" in v.lower())]
for v in fill_o3_vars:
    print(v)

print("\n=== single-instrument O3 q variables ===")
o3q_vars = [v for v in vars_all if v.lower().endswith("o3q")]
for v in o3q_vars:
    print(v)